In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import tqdm
import gzip
import json

import matplotlib
import pickle

import torch

from sklearn.manifold import TSNE

import sys; sys.path.insert(0, '..')
from sepsis_models.models.autoencoder import *
from sepsis_models.derived_information.neighbors import get_nearest_neighbors
from sepsis_models.derived_information.descriptive import DescriptiveInformation
from sepsis_models.derived_information.predictive import PredictiveActionDependentInformation, PredictiveActionIndependentInformation
from sepsis_models.derived_information.prescriptive import PrescriptiveOutcomeInformation, PrescriptivePeerInformation
from sepsis_models.derived_information.patient_info import create_state_dict, demog_features, state_features
from divisi.utils import convert_to_native_types

from google.cloud import storage

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Update these
data_dir = "../../sepsis-ai-modeling/250125_condensed_variables"
model_dir = "../../sepsis-ai-modeling/250125_condensed_variables/feature_weighted_autoencoder"

In [ ]:
AS_IS_COLUMNS = []
LOG_NORM_COLUMNS = [
    "SpO2", "BUN", "Creatinine", "AST", "ALT", 
    "Total Bili", "Direct Bili", "INR", "Input Last 24 h", "Output Last 24 h",
    "Hypertonic Crystalloid", "Hypotonic Crystalloid", "Isotonic Crystalloid",
    "Blood Products", "Isotonic Colloid",
    "Dopamine", "Epinephrine", "Norepinephrine", "Vasopressin", "Phenylephrine"
]

COMORBIDITIES = [
    'congestive_heart_failure', 'cardiac_arrhythmias', 'valvular_disease',
    'pulmonary_circulation', 'peripheral_vascular', 'hypertension',
    'paralysis', 'other_neurological', 'chronic_pulmonary',
    'diabetes_uncomplicated', 'diabetes_complicated', 'hypothyroidism',
    'renal_failure', 'liver_disease', 'peptic_ulcer',
    'aids', 'lymphoma', 'metastatic_cancer',
    'solid_tumor', 'rheumatoid_arthritis', 'coagulopathy',
    'obesity', 'weight_loss', 'fluid_electrolyte',
    'blood_loss_anemia', 'deficiency_anemias', 'alcohol_abuse',
    'drug_abuse', 'psychoses', 'depression'
]

EXCLUDE_COLUMNS = []

def format_states(df, scaler=None, all_columns=None):
    global AS_IS_COLUMNS, LOG_NORM_COLUMNS, EXCLUDE_COLUMNS
    dummies = pd.get_dummies(df, 
                        columns=[c for c in df.columns 
                                if pd.api.types.is_object_dtype(df[c].dtype) 
                                or pd.api.types.is_string_dtype(df[c].dtype) 
                                or isinstance(df[c].dtype, pd.CategoricalDtype)])
    if all_columns is not None:
        dummies = dummies.assign(**{c: np.zeros(len(dummies), dtype=np.uint8)
                                    for c in all_columns if c not in dummies.columns})
    if scaler is None:
        EXCLUDE_COLUMNS = [c for c in dummies.columns
                           if (('O2 Delivery Device' in c or c.startswith("Heart Rhythm")) and dummies[c].mean() < 0.01) or
                           c.endswith("Present")]
        dummies = dummies.drop(columns=EXCLUDE_COLUMNS)
        AS_IS_COLUMNS = ["id", "intime", "Invasive Ventilation", "Non-invasive Ventilation",
                 *[c for c in dummies.columns 
                   if "_" in c or 
                   " Present" in c or 
                   c.endswith("Current") or 
                   c.endswith("Ever") or 
                   (c.startswith("Positive") and c.endswith("Culture")) or 
                   c in COMORBIDITIES]]
        
        scaler = DataNormalization(dummies,
                                      as_is_columns=AS_IS_COLUMNS,
                                      log_norm_columns=LOG_NORM_COLUMNS,
                                      norm_columns=[c for c in dummies.columns if c not in AS_IS_COLUMNS + LOG_NORM_COLUMNS])
        return scaler.transform(dummies).assign(id=dummies['id'], intime=dummies['intime']), scaler, dummies.columns
    return scaler.transform(dummies.drop(columns=[c for c in EXCLUDE_COLUMNS if c in dummies.columns])).assign(id=dummies['id'], intime=dummies['intime'])
    
train_df, normalizer, all_cols = format_states(pd.read_csv(os.path.join(data_dir, "model_features", "inputs_train.csv"), index_col=0))
train_df = train_df.clip(-10, 10).assign(id=train_df['id'], intime=train_df['intime'])
val_df = format_states(pd.read_csv(os.path.join(data_dir, "model_features", "inputs_val.csv"), index_col=0), scaler=normalizer, all_columns=all_cols)
val_df = val_df.clip(-10, 10).assign(id=val_df['id'], intime=val_df['intime'])
test_df = format_states(pd.read_csv(os.path.join(data_dir, "model_features", "inputs_test.csv"), index_col=0), scaler=normalizer, all_columns=all_cols)
test_df = test_df.clip(-10, 10).assign(id=test_df['id'], intime=test_df['intime'])
print("Excluding columns:", EXCLUDE_COLUMNS)

train_uo = pd.read_csv(os.path.join(data_dir, "filters", "uo", "train.csv"), index_col=0).values[:,-1]
val_uo = pd.read_csv(os.path.join(data_dir, "filters", "uo", "val.csv"), index_col=0).values[:,-1]
test_uo = pd.read_csv(os.path.join(data_dir, "filters", "uo", "test.csv"), index_col=0).values[:,-1]
train_cmo = pd.read_csv(os.path.join(data_dir, "filters", "cmo", "train.csv"), index_col=0).values[:,-1]
val_cmo = pd.read_csv(os.path.join(data_dir, "filters", "cmo", "val.csv"), index_col=0).values[:,-1]
test_cmo = pd.read_csv(os.path.join(data_dir, "filters", "cmo", "test.csv"), index_col=0).values[:,-1]

# filter inputs to exclude patients who have less than 1 urine observation per day on average
train_mask = ~train_uo.astype(bool) & ~train_cmo.astype(bool) # pd.isna(train_y).groupby(train_df['id']).transform("sum") < max_missing
val_mask = ~val_uo.astype(bool) & ~val_cmo.astype(bool) # pd.isna(val_y).groupby(val_df['id']).transform("sum") < max_missing
test_mask = ~test_uo.astype(bool) & ~test_cmo.astype(bool) # pd.isna(test_y).groupby(test_df['id']).transform("sum") < max_missing

# filter to a maximum of 14 days per patient (84 timesteps), minimum of 12 hours (3 timesteps)
max_steps = np.round(np.quantile(train_df.groupby('id').size(), 0.95))
print('max steps:', max_steps)
min_days = 0.5
time_res = 4
train_mask &= ((train_df['intime'].groupby(train_df['id']).transform(lambda g: np.argsort(g)) < max_steps)
              & (pd.Series(train_mask, index=train_df.index).groupby(train_df['id']).transform("sum") >= min_days * 24 / time_res))
val_mask &= ((val_df['intime'].groupby(val_df['id']).transform(lambda g: np.argsort(g)) < max_steps)
            & (pd.Series(val_mask, index=val_df.index).groupby(val_df['id']).transform("sum") >= min_days * 24 / time_res))
test_mask &= ((test_df['intime'].groupby(test_df['id']).transform(lambda g: np.argsort(g)) < max_steps)
             & (pd.Series(test_mask, index=test_df.index).groupby(test_df['id']).transform("sum") >= min_days * 24 / time_res))

# TEMPORARY: sample test IDs
# np.random.seed(1234)
# test_sample_ids = np.random.choice(test_df['id'].unique(), size=2000, replace=False)
# test_mask &= test_df['id'].isin(test_sample_ids)

print("Before:", len(train_df), len(val_df), len(test_df))
train_df = train_df[train_mask].reset_index(drop=True)
val_df = val_df[val_mask].reset_index(drop=True)
test_df = test_df[test_mask].reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df))

In [ ]:
best_param_set = {'architecture': 'transformer', 
                  'nencoder': 4, 
                  'nhead': 8, 
                  'nhid': 128, 
                  'nembed': 32, 
                  'lr': 0.001, 
                  'dropout': 0.1, 
                  'lr_decay': 0.98, 
                  'n_warmup': 4, 
                  'mask_gamma': 1.01}
model_trainer = TimeSeriesAutoencoderTrainer(
    train_df,
    val_df,
    test_df,
    time_col='intime',
    device='cpu',
    checkpoint_path=os.path.join(model_dir, f"weighted_transformer.pt"),
    **best_param_set
)

# model_trainer.fit(epochs=50, patience=200, batch_size=32)
model_trainer.load_checkpoint()

In [ ]:
severity_scores_train = pd.read_csv(os.path.join(data_dir, "outcomes", "sofa", "train.csv"), index_col=0).iloc[:,-1].values
severity_scores_train = severity_scores_train[train_mask]
severity_scores_val = pd.read_csv(os.path.join(data_dir, "outcomes", "sofa", "val.csv"), index_col=0).iloc[:,-1].values
severity_scores_val = severity_scores_val[val_mask]
severity_scores_test = pd.read_csv(os.path.join(data_dir, "outcomes", "sofa", "test.csv"), index_col=0).iloc[:,-1].values
severity_scores_test = severity_scores_test[test_mask]

severity_cutoffs = np.quantile(severity_scores_train, np.linspace(0, 1, 9))
print("SOFA score cutoffs:", severity_cutoffs)
train_severity_quantiles = np.digitize(severity_scores_train, severity_cutoffs[:-1]) - 1
val_severity_quantiles = np.digitize(severity_scores_val, severity_cutoffs[:-1]) - 1
test_severity_quantiles = np.digitize(severity_scores_test, severity_cutoffs[:-1]) - 1

In [ ]:
def create_treatments(split):
    treatments = pd.concat([
        pd.read_csv(os.path.join(data_dir, "treatments", "fluids", f"{split}.csv"), index_col=0).drop(columns=["id", "intime"]),
        pd.read_csv(os.path.join(data_dir, "treatments", "vaso", f"{split}.csv"), index_col=0).drop(columns=["id", "intime"]),
    ], axis=1)
    treatments.columns = ["fluid", "vaso"]
    treatments['vaso'] = treatments['vaso'].fillna(0).astype(int)
    treatments['fluid'] = treatments['fluid'].fillna(0).astype(int)
    return treatments

treatments_train = create_treatments('train')[train_mask]
treatments_test = create_treatments('test')[test_mask]

In [ ]:
# example: nearest neighbors of the validation set instances in the training set
neighborhoods_dir = os.path.join(data_dir, "neighborhoods")
if not os.path.exists(neighborhoods_dir): os.mkdir(neighborhoods_dir)

if not os.path.exists(os.path.join(neighborhoods_dir, "train_neighbors.pkl")):
    train_encoded = model_trainer.encode(split='train')
    test_encoded = model_trainer.encode(split='test')
    train_neighb_train, train_neighb_dist_train = get_nearest_neighbors(train_encoded, 
                                                                    train_encoded, 
                                                                    train_df["id"].values, 
                                                                    reference_identical=True,
                                                                    stratify=(train_severity_quantiles, train_severity_quantiles))
    test_neighb_train, test_neighb_dist_train = get_nearest_neighbors(test_encoded, 
                                                                    train_encoded, 
                                                                    train_df["id"].values, 
                                                                    stratify=(test_severity_quantiles, train_severity_quantiles))

    with open(os.path.join(neighborhoods_dir, "train_neighbors.pkl"), "wb") as file:
        pickle.dump((train_neighb_train, train_neighb_dist_train), file)
    with open(os.path.join(neighborhoods_dir, "test_neighbors.pkl"), "wb") as file:
        pickle.dump((test_neighb_train, test_neighb_dist_train), file)
else:
    with open(os.path.join(neighborhoods_dir, "train_neighbors.pkl"), "rb") as file:
        train_neighb_train, train_neighb_dist_train = pickle.load(file)
    with open(os.path.join(neighborhoods_dir, "test_neighbors.pkl"), "rb") as file:
        test_neighb_train, test_neighb_dist_train = pickle.load(file)
    

In [ ]:
# Preprocess slicing variables
train_slicing_vars = pd.read_csv(os.path.join(data_dir, "slicing", "inputs_train.csv"), index_col=0)[train_mask].reset_index(drop=True)
eval_slicing_vars = pd.read_csv(os.path.join(data_dir, "slicing", "inputs_test.csv"), index_col=0)[test_mask].reset_index(drop=True)
COLUMNS_TO_DROP = [
    "Gender"
]
train_slicing_vars = train_slicing_vars.drop(columns=COLUMNS_TO_DROP)
eval_slicing_vars = eval_slicing_vars.drop(columns=COLUMNS_TO_DROP)

for c in eval_slicing_vars.columns:
    try:
        if len(np.unique(eval_slicing_vars[c])) == 2 and (np.unique(eval_slicing_vars[c]) == np.array([0, 1])).all():
            print(c)
            eval_slicing_vars[c] = eval_slicing_vars[c].astype(int).replace({0: 'No', 1: 'Yes'})
            train_slicing_vars[c] = train_slicing_vars[c].astype(int).replace({0: 'No', 1: 'Yes'})
    except: pass

eval_slicing_vars = eval_slicing_vars.fillna('Missing')
train_slicing_vars = train_slicing_vars.fillna('Missing')

# Uploading Patient Data

In [ ]:
gcs_client = storage.Client.from_service_account_json('../../ai-clinician-94fd8d5e44f2.json')
bucket = gcs_client.bucket("sepsis-challenging-cases")

In [ ]:
cases_to_select = pd.read_csv(os.path.join(data_dir, "interesting_prediction_cases.csv"))
matching_notes = pd.read_csv("../../AI-Clinician-MIMICIV/data/intermediates/patient_matched_notes.csv")
cases_to_select = cases_to_select[cases_to_select['id'].isin(matching_notes['icustayid'])]

reasoning_mask = test_df['id'].astype(int).isin(cases_to_select['id'].drop_duplicates())
print(reasoning_mask.sum())

In [ ]:
patient_table = {}
dataset_name = "weighted" # "contrastive"

# Patient Scenarios

## Scenario 1

In the previous epoch, given a lot of fluid. Not intubated yet, in (mild) renal failure (creatinine slightly high), mAP on the low side, 65 or lower, but no pressors. Need to decide, do I start a pressor or give more fluid?
+ History of heart disease/CHF
+ if first or second epoch it would be better

## Scenario 2

When to add the second pressor? Maybe the patient is intubated and on a pressor, but their mAP is on the low side. Do I give fluid or add the second pressor?

## Scenario 3

The patient is intubated and on a low-dose pressor. Decent amount of FiO2 (50% or higher). 
+ this is a case on morning rounds
+ Renal failure and heart failure are nice but not essential
Want to get the patient off the ventilator, which would either require diuresing or giving fluid to get off the pressor. Maybe a reasonable decision is to do nothing?

## Scenario 4

Something where we have to decide between a little bit of fluid and a lot of fluid.

In [ ]:
hr_test_states = pd.read_csv(os.path.join(data_dir, "human_readable", "inputs_test.csv"), index_col=0)[test_mask]
hr_test_states['id'] = hr_test_states['id'].astype(int)

boolean_columns = []
for c in hr_test_states.columns:
    try:
        if len(np.unique(hr_test_states[c])) == 2 and (np.unique(hr_test_states[c]) == np.array([0, 1])).all():
            boolean_columns.append(c)
    except: pass
hr_test_states[boolean_columns] = hr_test_states[boolean_columns].astype(pd.BooleanDtype())

hr_test_states = pd.merge(hr_test_states, matching_notes.drop_duplicates('icustayid'), how='left', left_on='id', right_on='icustayid')
# hr_test_states['complex_flag'] = ~pd.isna(pd.merge(hr_test_states[['id', 'intime']], 
#                                                     cases_to_select.rename(columns={'intime': 'reasoning_time'}), 
#                                                     how='left', 
#                                                     left_on=['id', 'intime'], 
#                                                     right_on=['id', 'reasoning_time'])['reasoning_time'])
print("Final:", len(hr_test_states), len(hr_test_states["id"].unique()))

raw_test_states = pd.read_csv(os.path.join(data_dir, "model_features", "inputs_test.csv"), index_col=0)[test_mask]
raw_test_states['id'] = raw_test_states['id'].astype(int)
raw_test_states = pd.merge(raw_test_states, matching_notes.drop_duplicates('icustayid'), how='left', left_on='id', right_on='icustayid')
print("Final:", len(raw_test_states), len(raw_test_states["id"].unique()))


In [ ]:
i = 0
for name, group in tqdm.tqdm(hr_test_states[reasoning_mask].groupby("id")):
    demog = create_state_dict(demog_features, group.iloc[0])
    result = {
        "id": name,
        "num_timesteps": len(group),
        # "interesting_indexes": np.argwhere(group['complex_flag'].values).flatten().tolist(),
        "data": {
            "Demographics": {
                "data": demog
            },
            "State": {
                "timesteps": [
                    { "time": r["intime"], "data": create_state_dict(state_features, r) }
                    for _, r in group.iterrows()
                ]
            },
            "Notes": {
                "data": [
                    { "name": "Discharge Summary", "value": group.iloc[0]['target'] }
                ] if not pd.isna(group.iloc[0]['target']) else []
            }
        }
    }
    patient_table.setdefault("id", []).extend([name for _ in range(len(group))])
    patient_table.setdefault("timestep", []).extend(list(range(len(group))))
    patient_table.setdefault("age", []).extend(group['age'].values.tolist())
    patient_table.setdefault("gender", []).extend(np.where(group['gender_female'], 'Female', 'Male').tolist())
    patient_table.setdefault("Fluid Last 4 h", []).extend(raw_test_states.loc[group.index][['Isotonic Crystalloid', 
                 'Isotonic Colloid', 
                 'Hypotonic Crystalloid', 
                 'Hypertonic Crystalloid', 
                 'Blood Products']].sum(axis=1).values.tolist())
    patient_table.setdefault("Vasopressors", []).extend(((group[["vaso_Norepinephrine", "vaso_Epinephrine", "vaso_Vasopressin", "vaso_Phenylephrine", "vaso_Dopamine"]] >= 0.01).sum(axis=1)).values.tolist())

    patient_table.setdefault("Intubated", []).extend(group['Invasive Ventilation'].values.tolist())
    patient_table.setdefault("FiO2", []).extend(group['FiO2'].where(group['FiO2 Present'], None).values.tolist())
    patient_table.setdefault("CHF History", []).extend(group['congestive_heart_failure'].values.tolist())
    patient_table.setdefault("Creatinine", []).extend(group['Creatinine'].values.tolist())
    patient_table.setdefault("mAP", []).extend(group['mAP'].values.tolist())
    
    if dataset_name is not None:
        content = gzip.compress(json.dumps(convert_to_native_types(result), separators=(',',':')).encode('ascii'))
        with bucket.blob(f'{dataset_name}/states/{name}.json.gz').open('wb') as file: 
            file.write(content)
    
    i += 1
    # if i == 50: break

In [ ]:
# Write a file with the IDs of all patients
if dataset_name is not None:
    blob = bucket.blob(f"{dataset_name}/meta.json")
    with blob.open("w") as f:
        json.dump({
            "patient_ids": cases_to_select['id'].tolist() #[:50]
        }, f)
        
    with bucket.blob("meta.json").open("w") as file:
        json.dump({"datasets": ["reasoning", "contrastive", "weighted"]}, file)

In [ ]:
def write_patient_insight(data_key, name, group, insight_fn):
    if dataset_name is None: return
    name = int(name)
    result = {
        "id": name,
        "data": {
            data_key: {
                "timesteps": [
                    { "time": r["intime"], "data": insight_fn(idx, r) or {} }
                    for idx, r in group.iterrows()
                ]
            }
        }
    }
    content = gzip.compress(json.dumps(convert_to_native_types(result), separators=(',',':')).encode('ascii'))
    with bucket.blob(f'{dataset_name}/{data_key}/{name}.json.gz').open('wb') as file: 
        file.write(content)
    return [x["data"] for x in result["data"][data_key]["timesteps"]]

In [ ]:
descriptive_info = DescriptiveInformation(train_slicing_vars)

i = 0
for name, group in tqdm.tqdm(eval_slicing_vars[reasoning_mask].groupby("id")):
    results = write_patient_insight("descriptive", name, group, lambda idx, r: {
        "similar": descriptive_info.get_most_similar_features(test_neighb_train[idx], r),
        "different": descriptive_info.get_most_different_features(test_neighb_train[idx], r)
    })
    
    patient_table.setdefault("similar", []).extend([", ".join(x["feature"] for x in r["similar"])
                                                    for r in results])
    patient_table.setdefault("different", []).extend([", ".join(x["feature"] for x in r["different"])
                                                    for r in results])
    i += 1
    # if i == 50: break

In [ ]:
train_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "vaso", "train.csv"), index_col=0).iloc[:,-1].values[train_mask]
eval_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "vaso", "test.csv"), index_col=0).iloc[:,-1].values[test_mask]

vaso_independent = PredictiveActionIndependentInformation(train_outcome, train_severity_quantiles, severity_cutoffs)

i = 0
for name, group in tqdm.tqdm(test_df[reasoning_mask].groupby("id")):
    results = write_patient_insight("predictive_vaso_independent", name, group, 
                          lambda idx, r: {
                              **vaso_independent.get_prediction(
                              test_neighb_train[idx], 
                              test_severity_quantiles[idx], 
                              true_outcome=eval_outcome[idx]),
                              "explanation": descriptive_info.get_most_similar_features(test_neighb_train[idx], eval_slicing_vars.iloc[idx])
                            })
    
    patient_table.setdefault("pred_vaso_ind_pred", []).extend([p["prediction_percentage"] for p in results])
    patient_table.setdefault("pred_vaso_ind_true", []).extend([p["ground_truth"] for p in results])
    i += 1
    # if i == 50: break

In [ ]:
train_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "morta", "train.csv"), index_col=0).iloc[:,-1].values[train_mask]
eval_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "morta", "test.csv"), index_col=0).iloc[:,-1].values[test_mask]

morta_independent = PredictiveActionIndependentInformation(train_outcome, train_severity_quantiles, severity_cutoffs)

i = 0
for name, group in tqdm.tqdm(test_df[reasoning_mask].groupby("id")):
    results = write_patient_insight("predictive_morta_independent", name, group, 
                          lambda idx, r: {
                              **morta_independent.get_prediction(
                              test_neighb_train[idx], 
                              test_severity_quantiles[idx], 
                              true_outcome=eval_outcome[idx]),
                              "explanation": descriptive_info.get_most_similar_features(test_neighb_train[idx], eval_slicing_vars.iloc[idx])
                            })

    patient_table.setdefault("pred_morta_ind_pred", []).extend([p["prediction_percentage"] for p in results])
    patient_table.setdefault("pred_morta_ind_true", []).extend([p["ground_truth"] for p in results])
    i += 1
    # if i == 50: break

In [ ]:
train_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "vaso", "train.csv"), index_col=0).iloc[:,-1].values[train_mask]
eval_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "vaso", "test.csv"), index_col=0).iloc[:,-1].values[test_mask]

vaso_dependent = PredictiveActionDependentInformation(train_neighb_train, train_df['id'].astype(int), train_outcome, treatments_train.values, severity_cutoffs)

i = 0
for name, group in tqdm.tqdm(test_df[reasoning_mask].groupby("id")):
    results = write_patient_insight("predictive_vaso_dependent", name, group, 
                          lambda idx, r: {
                              **vaso_dependent.get_prediction(
                              test_neighb_train[idx], 
                              test_severity_quantiles[idx], 
                              true_outcome=eval_outcome[idx],
                              last_treatment=treatments_test.values[idx - 1] if idx > group.index[0] else None),
                              "explanation": descriptive_info.get_most_similar_features(test_neighb_train[idx], eval_slicing_vars.iloc[idx])
                            })

    patient_table.setdefault("pred_vaso_dep_n_sig", []).extend([sum(p["prediction"]["pvalue"] <= 0.05 for p in r["predictions"]) for r in results])
    patient_table.setdefault("pred_vaso_dep_largest_impr", []).extend([max(p["prediction"]["mean"] - r["average"]["prediction"]["mean"] for p in r["predictions"]) for r in results])
    patient_table.setdefault("pred_vaso_dep_largest_red", []).extend([min(p["prediction"]["mean"] - r["average"]["prediction"]["mean"] for p in r["predictions"]) for r in results])
    patient_table.setdefault("pred_vaso_dep_avg", []).extend([p["average"]["prediction"]["mean"] for p in results])
    i += 1
    # if i == 50: break

In [ ]:
train_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "morta", "train.csv"), index_col=0).iloc[:,-1].values[train_mask]
eval_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "morta", "test.csv"), index_col=0).iloc[:,-1].values[test_mask]

morta_dependent = PredictiveActionDependentInformation(train_neighb_train, train_df['id'].astype(int), train_outcome, treatments_train.values, severity_cutoffs)

i = 0
for name, group in tqdm.tqdm(test_df[reasoning_mask].groupby("id")):
    results = write_patient_insight("predictive_morta_dependent", name, group, 
                        lambda idx, r: {
                            **morta_dependent.get_prediction(
                            test_neighb_train[idx], 
                            test_severity_quantiles[idx], 
                            true_outcome=eval_outcome[idx],
                            last_treatment=treatments_test.values[idx - 1] if idx > group.index[0] else None),
                            "explanation": descriptive_info.get_most_similar_features(test_neighb_train[idx], eval_slicing_vars.iloc[idx])
                        })

    patient_table.setdefault("pred_morta_dep_n_sig", []).extend([sum(p["prediction"]["pvalue"] <= 0.05 for p in r["predictions"]) for r in results])
    patient_table.setdefault("pred_morta_dep_avg", []).extend([p["average"]["prediction"]["mean"] for p in results])
    patient_table.setdefault("pred_morta_dep_largest_impr", []).extend([max(p["prediction"]["mean"] - r["average"]["prediction"]["mean"] for p in r["predictions"]) for r in results])
    patient_table.setdefault("pred_morta_dep_largest_red", []).extend([min(p["prediction"]["mean"] - r["average"]["prediction"]["mean"] for p in r["predictions"]) for r in results])

    i += 1
    # if i == 50: break

In [ ]:
clinician_rec = PrescriptivePeerInformation(treatments_train.values, severity_cutoffs)
i = 0
for name, group in tqdm.tqdm(test_df[reasoning_mask].groupby("id")):
    results = write_patient_insight("prescriptive_peer", name, group, 
                        lambda idx, r: {
                            **clinician_rec.get_recommendation(
                            test_neighb_train[idx], 
                            test_severity_quantiles[idx],
                            true_treatment=treatments_test.values[idx],
                            last_treatment=treatments_test.values[idx - 1] if idx > group.index[0] else None),
                            "explanation": descriptive_info.get_most_similar_features(test_neighb_train[idx], eval_slicing_vars.iloc[idx])
                        })

    patient_table.setdefault("presc_peer_volume_consistent", []).extend([r["prediction"][0]["pred"]["consistent"] for r in results])
    patient_table.setdefault("presc_peer_vaso_consistent", []).extend([r["prediction"][1]["pred"]["consistent"] for r in results])
    patient_table.setdefault("presc_peer_volume_matches", []).extend([r["prediction"][0]["pred"]["consistent"] and r["prediction"][0]["pred"]["choice_idx"] == r["ground_truth"][0]["label_idx"]
                                                                      for r in results])
    patient_table.setdefault("presc_peer_vaso_matches", []).extend([r["prediction"][1]["pred"]["consistent"] and r["prediction"][1]["pred"]["choice_idx"] == r["ground_truth"][1]["label_idx"]
                                                                      for r in results])
    patient_table.setdefault("volume_decision", []).extend([r["ground_truth"][0]["label_idx"]
                                                            for r in results])
    patient_table.setdefault("vaso_decision", []).extend([r["ground_truth"][1]["label_idx"]
                                                          for r in results])
    i += 1
    # if i == 50: break

In [ ]:
train_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "morta", "train.csv"), index_col=0).iloc[:,-1].values[train_mask]
eval_outcome = pd.read_csv(os.path.join(data_dir, "outcomes", "morta", "test.csv"), index_col=0).iloc[:,-1].values[test_mask]

ai_clinician = PrescriptiveOutcomeInformation(train_neighb_train, train_df['id'].astype(int), train_outcome, treatments_train.values, severity_cutoffs)

i = 0
def make_ai_clinician(idx, r):
    rec = ai_clinician.get_recommendation(
                            test_neighb_train[idx], 
                            test_severity_quantiles[idx],
                            true_treatment=treatments_test.values[idx],
                            last_treatment=treatments_test.values[idx - 1] if idx > group.index[0] else None)
    if not rec: return None
    return {**rec,
            "explanation": descriptive_info.get_most_similar_features(test_neighb_train[idx], eval_slicing_vars.iloc[idx])}
for name, group in tqdm.tqdm(test_df[reasoning_mask].groupby("id")):
    results = write_patient_insight("prescriptive_outcome", name, group, make_ai_clinician)

    patient_table.setdefault("has_presc_morta_rec", []).extend([r is not None for r in results])
    patient_table.setdefault("presc_morta_volume", []).extend([r["recommendation"][0]["value"] if r else ""
                                                                      for r in results])
    patient_table.setdefault("presc_morta_vaso", []).extend([r["recommendation"][1]["value"] if r else ""
                                                                      for r in results])

    i += 1
    # if i == 50: break

In [ ]:
patient_table = pd.DataFrame(patient_table)
for c in patient_table.columns:
    if set(patient_table[c].unique()) == set(['No', 'Yes']):
        patient_table[c] = patient_table[c].replace({'No': 0, 'Yes': 1}).astype(bool)
        print(c)
patient_table

In [ ]:
patient_table.to_csv(os.path.join(data_dir, f"patient_rec_summaries_{dataset_name}_supp.csv"), index=False)